In [1]:
from zipfile import ZipFile
import os
import numpy as np
import math
import shutil
from keras.layers import Conv2D, MaxPool2D, Dropout, Flatten, Dense
from keras.models import Sequential
from keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ModelCheckpoint, EarlyStopping
from keras.models import load_model
from keras.preprocessing.image import load_img, img_to_array

In [2]:
# Unzipping the dataset
dataset = "Brain_Tumor_Dataset.zip"
with ZipFile(dataset, 'r') as zip:
    zip.extractall()
    print("Dataset extracted successfully!")

Dataset extracted successfully!


In [3]:
# Organizing the dataset
ROOT_DIR = 'Brain_Tumor_Dataset'
images = {dir: len(os.listdir(os.path.join(ROOT_DIR, dir))) for dir in os.listdir(ROOT_DIR)}


In [4]:
if not os.path.exists("train"):
    os.mkdir("train")
    for dir in os.listdir(ROOT_DIR):
        os.makedirs(f"train/{dir}", exist_ok=True)
        for img in np.random.choice(
            os.listdir(os.path.join(ROOT_DIR, dir)),
            size=(math.floor(0.7 * images[dir])),
            replace=False,
        ):
            O = os.path.join(ROOT_DIR, dir, img)
            D = os.path.join("train", dir, img)
            shutil.copy(O, D)
else:
    print("Train directory already exists.")

Train directory already exists.


In [5]:
if not os.path.exists("test"):
    os.mkdir("test")
    for dir in os.listdir(ROOT_DIR):
        os.makedirs(f"test/{dir}", exist_ok=True)
        for img in np.random.choice(
            os.listdir(os.path.join(ROOT_DIR, dir)),
            size=(math.floor(0.3 * images[dir])),
            replace=False,
        ):
            O = os.path.join(ROOT_DIR, dir, img)
            D = os.path.join("test", dir, img)
            shutil.copy(O, D)
else:
    print("Test directory already exists.")

Test directory already exists.


In [6]:
# Model Definition
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPool2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPool2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPool2D((2, 2)),
    Dropout(0.5),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

C:\Users\SAKSHAM CHAND\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# Data Generators
train_images = ImageDataGenerator(
    rescale=1.0 / 255, shear_range=0.2, zoom_range=0.2, horizontal_flip=True
)
train_data = train_images.flow_from_directory(
    directory="train", target_size=(224, 224), batch_size=32, class_mode='binary'
)

test_images = ImageDataGenerator(rescale=1.0 / 255)
test_data = test_images.flow_from_directory(
    directory="test", target_size=(224, 224), batch_size=32, class_mode='binary'
)

Found 3816 images belonging to 2 classes.
Found 1629 images belonging to 2 classes.


In [9]:
# Callbacks
es = EarlyStopping(monitor="val_accuracy", patience=5, verbose=1, mode='max')
mc = ModelCheckpoint(
    filepath="./Brain_Tumor.keras",
    monitor="val_accuracy",
    verbose=1,
    save_best_only=True,
    mode='max'
)
callbacks = [es, mc]

In [10]:
# Model Training
history = model.fit(
    train_data,
    steps_per_epoch=len(train_data),
    epochs=30,
    validation_data=test_data,
    validation_steps=len(test_data),
    callbacks=callbacks
)

C:\Users\SAKSHAM CHAND\AppData\Roaming\Python\Python312\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7225 - loss: 0.5939

C:\Users\SAKSHAM CHAND\AppData\Roaming\Python\Python312\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()



Epoch 1: val_accuracy improved from -inf to 0.92818, saving model to ./Brain_Tumor.keras
120/120 ━━━━━━━━━━━━━━━━━━━━ 139s 1s/step - accuracy: 0.7234 - loss: 0.5923 - val_accuracy: 0.9282 - val_loss: 0.1765
Epoch 2/30
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 842ms/step - accuracy: 0.9191 - loss: 0.2200
Epoch 2: val_accuracy did not improve from 0.92818
120/120 ━━━━━━━━━━━━━━━━━━━━ 111s 919ms/step - accuracy: 0.9191 - loss: 0.2199 - val_accuracy: 0.8920 - val_loss: 0.2539
Epoch 3/30
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 855ms/step - accuracy: 0.9197 - loss: 0.2044
Epoch 3: val_accuracy improved from 0.92818 to 0.94475, saving model to ./Brain_Tumor.keras
120/120 ━━━━━━━━━━━━━━━━━━━━ 114s 947ms/step - accuracy: 0.9198 - loss: 0.2043 - val_accuracy: 0.9448 - val_loss: 0.1284
Epoch 4/30
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 861ms/step - accuracy: 0.9462 - loss: 0.1497
Epoch 4: val_accuracy improved from 0.94475 to 0.95335, saving model to ./Brain_Tumor.keras
120/120 ━━━━━━━━━━━━━━━━━━━━ 114s 944ms/step - accu

In [12]:
# Save the model
model.save('Brain_Tumor.keras')

In [13]:
# Prediction
def predict_brain_tumor(image_path):
    image = load_img(image_path, target_size=(224, 224))
    image = img_to_array(image) / 255.0
    image = np.expand_dims(image, axis=0)
    pred = model.predict(image)[0][0]
    if pred > 0.5:
        print("Tumor Detected")
    else:
        print("No Tumor Detected")
    print(f"Prediction Score: {pred}")